# Final Salient Object Detection Demo

This notebook is the clean presentation version of the working SOD pipeline. It does not retrain the model. It loads the trained checkpoint, uses the selected threshold `0.40`, and demonstrates saliency-mask inference with overlays and timing.

## 1. Environment And Paths

The notebook supports Google Colab and a local/GitHub checkout. If this notebook is opened directly in Colab without the full repository, the setup cell clones the GitHub repo first so imports from `src/` work correctly.

In [ ]:
import os
import random
import subprocess
import sys
import time
from pathlib import Path

REPO_URL = 'https://github.com/florjete/salient-object-detection-cnn.git'
REPO_NAME = 'salient-object-detection-cnn'
IN_COLAB = 'COLAB_GPU' in os.environ

# If this notebook is opened by itself in Colab, clone the full repository
# so src/sod_model.py, src/utils.py, and src/evaluate.py are available.
if IN_COLAB and not Path('src').exists():
    repo_path = Path('/content') / REPO_NAME
    if not repo_path.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(repo_path)], check=True)
    os.chdir(repo_path)

print('Current working directory:', Path.cwd())
print('src exists:', Path('src').exists())

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torchvision.transforms import functional as TF

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    CHECKPOINT_PATH = Path('/content/drive/MyDrive/sod_data/checkpoints/best_model.pth')
    IMAGE_DIR = Path('/content/drive/MyDrive/sod_data/MSRA10K/images')
    MASK_DIR = Path('/content/drive/MyDrive/sod_data/MSRA10K/masks')
else:
    CHECKPOINT_PATH = Path('checkpoints/best_model.pth')
    IMAGE_DIR = Path('data/MSRA10K/images')
    MASK_DIR = Path('data/MSRA10K/masks')

PROJECT_ROOT = Path.cwd()
src_candidates = [
    PROJECT_ROOT / 'src',
    PROJECT_ROOT.parent / 'src',
    Path('/content/salient-object-detection-cnn/src'),
]
SRC_DIR = next((path for path in src_candidates if path.exists()), src_candidates[0])
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGE_SIZE = 128

# Threshold 0.40 was selected because it achieved the best validation F1-score
# in the working one_day_sod_pipeline.ipynb threshold search.
THRESHOLD = 0.40

print('Running in Colab:', IN_COLAB)
print('Device:', DEVICE)
print('Checkpoint path:', CHECKPOINT_PATH)
print('Image directory:', IMAGE_DIR)
print('Mask directory:', MASK_DIR)
print('Selected threshold:', THRESHOLD)

## 2. Import Project Code

The demo reuses the repository implementation where possible. The model architecture is `ImprovedSODNet`, matching the working notebook and trained checkpoint.

In [ ]:
from sod_model import ImprovedSODNet
from utils import BCEIoULoss, compute_metrics, load_checkpoint, overlay_mask, predict_image, set_seed

set_seed(42)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f'Checkpoint not found: {CHECKPOINT_PATH}. '
        'Place best_model.pth at this path or update CHECKPOINT_PATH.'
    )
if not IMAGE_DIR.exists():
    raise FileNotFoundError(f'Image directory not found: {IMAGE_DIR}')
if not MASK_DIR.exists():
    raise FileNotFoundError(f'Mask directory not found: {MASK_DIR}')

## 3. Load Trained Model

The checkpoint is loaded directly. No training is performed in this demo notebook.

In [ ]:
model = ImprovedSODNet().to(DEVICE)
checkpoint = load_checkpoint(CHECKPOINT_PATH, model, device=DEVICE)
model.eval()

print('Loaded checkpoint successfully.')
print('Checkpoint epoch:', checkpoint.get('epoch', 'not stored'))
print('Checkpoint model name:', checkpoint.get('model_name', 'not stored'))

## 4. Final Reported Metrics

These are the selected demo results from the improved from-scratch model. They are reported honestly as a strong baseline, not as state-of-the-art SOD performance.

In [ ]:
reported_metrics = {
    'IoU': 0.6470,
    'Precision': 0.7582,
    'Recall': 0.8267,
    'F1': 0.7895,
}

print('Final selected demo metrics')
print(f'Threshold: {THRESHOLD:.2f}')
for name, value in reported_metrics.items():
    print(f'{name}: {value:.4f}')

## 5. Build Sample Image-Mask Pairs

The dataset uses matching filenames such as `101.jpg` and `101.png`. This cell verifies matching pairs for demo visualization.

In [ ]:
image_paths = sorted([*IMAGE_DIR.glob('*.jpg'), *IMAGE_DIR.glob('*.jpeg')])
mask_paths = sorted(MASK_DIR.glob('*.png'))
image_by_stem = {path.stem: path for path in image_paths}
mask_by_stem = {path.stem: path for path in mask_paths}
matched_stems = sorted(set(image_by_stem) & set(mask_by_stem))

if not matched_stems:
    raise RuntimeError('No matched image/mask pairs found. Check IMAGE_DIR and MASK_DIR.')

pairs = [(image_by_stem[stem], mask_by_stem[stem]) for stem in matched_stems]
print('Image files:', len(image_paths))
print('Mask files:', len(mask_paths))
print('Matched pairs:', len(pairs))

## 6. Inference Helper

This helper mirrors the working notebook preprocessing: resize to `128x128`, convert RGB image to a `0..1` tensor, run the model, threshold at `0.40`, and measure inference time.

In [ ]:
def run_demo_inference(model, image_path, device, threshold=0.40, image_size=128):
    image = Image.open(image_path).convert('RGB')
    resized = TF.resize(image, (image_size, image_size), interpolation=TF.InterpolationMode.BILINEAR)
    tensor = TF.to_tensor(resized).unsqueeze(0).to(device)

    model.eval()
    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        pred = model(tensor)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    probability_mask = pred[0, 0].detach().cpu()
    binary_mask = (probability_mask >= threshold).float()
    return resized, probability_mask, binary_mask, elapsed

## 7. Sample Predictions

The visualization shows the required demo outputs: input image, ground-truth saliency mask, predicted mask, and overlay.

In [ ]:
random.seed(42)
sample_pairs = random.sample(pairs, k=min(4, len(pairs)))

plt.figure(figsize=(16, 12))
inference_times = []

for idx, (image_path, mask_path) in enumerate(sample_pairs):
    image, probability_mask, binary_mask, elapsed = run_demo_inference(
        model,
        image_path,
        DEVICE,
        threshold=THRESHOLD,
        image_size=IMAGE_SIZE,
    )
    inference_times.append(elapsed)
    mask = Image.open(mask_path).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), resample=Image.Resampling.NEAREST)
    image_tensor = TF.to_tensor(image)

    plt.subplot(4, len(sample_pairs), idx + 1)
    plt.imshow(image)
    plt.title('Input image')
    plt.axis('off')

    plt.subplot(4, len(sample_pairs), len(sample_pairs) + idx + 1)
    plt.imshow(mask, cmap='gray')
    plt.title('Ground truth')
    plt.axis('off')

    plt.subplot(4, len(sample_pairs), 2 * len(sample_pairs) + idx + 1)
    plt.imshow(binary_mask, cmap='gray')
    plt.title('Predicted mask')
    plt.axis('off')

    plt.subplot(4, len(sample_pairs), 3 * len(sample_pairs) + idx + 1)
    plt.imshow(overlay_mask(image_tensor, binary_mask))
    plt.title('Overlay')
    plt.axis('off')

plt.tight_layout()
plt.show()

for (image_path, _), elapsed in zip(sample_pairs, inference_times):
    print(f'{image_path.name}: {elapsed * 1000:.2f} ms per image')
print(f'Average inference time: {np.mean(inference_times) * 1000:.2f} ms per image')

## 8. Single-Image Demo

This cell runs one image and prints inference time clearly for presentation.

In [ ]:
demo_image_path, demo_mask_path = sample_pairs[0]
demo_image, demo_probability, demo_binary, demo_elapsed = run_demo_inference(
    model,
    demo_image_path,
    DEVICE,
    threshold=THRESHOLD,
    image_size=IMAGE_SIZE,
)

print('Demo image:', demo_image_path)
print(f'Inference time per image: {demo_elapsed * 1000:.2f} ms')

demo_mask = Image.open(demo_mask_path).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), resample=Image.Resampling.NEAREST)
demo_tensor = TF.to_tensor(demo_image)

plt.figure(figsize=(12, 3))
plt.subplot(1, 4, 1)
plt.imshow(demo_image)
plt.title('Input')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(demo_mask, cmap='gray')
plt.title('Ground truth')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(demo_binary, cmap='gray')
plt.title('Prediction')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(overlay_mask(demo_tensor, demo_binary))
plt.title('Overlay')
plt.axis('off')

plt.tight_layout()
plt.show()

## 9. Save Demo Outputs

This optional final cell saves one metrics file and one visualization figure. In Colab, they are written to Google Drive under `sod_data/outputs/`; locally, they are written to the repository `outputs/` folder.

In [ ]:
import json

if IN_COLAB:
    OUTPUT_DIR = Path('/content/drive/MyDrive/sod_data/outputs')
else:
    OUTPUT_DIR = Path('outputs')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metrics_to_save = {
    'threshold': THRESHOLD,
    'iou': 0.6470,
    'precision': 0.7582,
    'recall': 0.8267,
    'f1': 0.7895,
}

with open(OUTPUT_DIR / 'final_metrics.json', 'w') as file:
    json.dump(metrics_to_save, file, indent=2)

plt.figure(figsize=(12, 3))
plt.subplot(1, 4, 1)
plt.imshow(demo_image)
plt.title('Input')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(demo_mask, cmap='gray')
plt.title('Ground truth')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(demo_binary, cmap='gray')
plt.title('Prediction')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(overlay_mask(demo_tensor, demo_binary))
plt.title('Overlay')
plt.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'demo_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print('Saved metrics to:', OUTPUT_DIR / 'final_metrics.json')
print('Saved figure to:', OUTPUT_DIR / 'demo_predictions.png')